In [1]:
from ultralytics import YOLO, settings

settings.update({"mlflow": False})

model =  YOLO("yolo11n-seg.pt")
train = model.train(
    data="../dataset_collection/roboflow/segmentation/segmentation_mask_leaf_and_lession/data.yaml",
    epochs=5,
    imgsz=640,
    batch=16,
    project="training_segmentation_to_know_the_dict_name_resutls/yolo_v12",
    name="training",
    exist_ok=True

)
# val = model.val(
#     data="../dataset_collection/roboflow/segmentation/segmentation_mask_leaf_and_lession/data.yaml",
#     imgsz=640,
#     project="training_segmentation_to_know_the_dict_name_resutls/yolo_v12",
#     name="test",
#     split="test"
# )

New https://pypi.org/project/ultralytics/8.3.240 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.231  Python-3.12.12 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5070 Ti, 16303MiB)
engine\trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../dataset_collection/roboflow/segmentation/segmentation_mask_leaf_and_lession/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=5, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-seg.pt, momentum=0

In [10]:
precision=train.box.p.tolist()
recall=train.box.r.tolist()
mAP50_box = train.box.ap50.tolist()
mAP50_95_box = train.box.maps.tolist()
mAP50_mask = train.seg.ap50.tolist()
mAP50_95_mask = float(train.seg.maps)

TypeError: only length-1 arrays can be converted to Python scalars

In [12]:
def save_class_metrics(val_results, mode):
    # Ambil nama class
    names = val_results.names

    # Data per-class
    data = {
        "class": [names[i] for i in range(len(names))],
        "precision": val_results.box.p.tolist(),
        "recall": val_results.box.r.tolist(),
        "mAP50-box": val_results.box.ap50.tolist(),
        "mAP50-95-box": val_results.box.maps.tolist(),
        "mAP50-mask": val_results.seg.ap50.tolist(),
        "mAP50-95-mask": val_results.seg.maps.tolist(),
    }

    # Tambahkan baris agregat "all"
    all_row = {
        "class": "all",
        "precision": float(val_results.box.mp),
        "recall": float(val_results.box.mr),
        "mAP50-box": float(val_results.box.map50),
        "mAP50-95-box": float(val_results.box.map),
        "mAP50-mask": float(val_results.seg.map50),
        "mAP50-95-mask": float(val_results.seg.map),
    }

    for key in data.keys():
        data[key].insert(0, all_row[key])

    # Buat DataFrame dan simpan
    df = pd.DataFrame(data)
    csv_path = os.path.join(str(val_results.save_dir), f"class_metrics_{mode}.csv")
    df.to_csv(csv_path, index=False)

    return csv_path

In [15]:
import pandas as pd
import os
save_class_metrics(train, "train")

'C:\\Users\\abiyamf\\Documents\\Code Program\\Thesis\\random_experiment\\training_segmentation_to_know_the_dict_name_resutls\\yolo_v12\\training\\class_metrics_train.csv'